# Step 2 Solution Guide: Item Selection, Cleaning, and EDA

Step 2 turns extracted 10-K item files into a clean corpus. This version expects the improved Step 1 files from 2020 through the latest available filing year. It keeps ticker, company, year, NAICS, and item metadata attached to every cleaned file so the embedding and RAG notebooks can cite evidence correctly.

Good cleaning does not mean deleting everything difficult. It means removing obvious noise while preserving company, year, item, and business meaning.


## 1. Path Setup


In [ ]:
from pathlib import Path
import pandas as pd

# In Google Colab, this is the expected project folder.
# If you run locally, replace this path with Path.cwd() / "project_sec10k_rag".
BASE_DIR = Path("/content/drive/MyDrive/project_sec10k_rag")

DATA_DIR = BASE_DIR / "data"
OUTPUTS_DIR = DATA_DIR / "outputs"
REPORTS_DIR = BASE_DIR / "reports"

for folder in [DATA_DIR, OUTPUTS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE_DIR)

EXTRACTED_ITEMS_DIR = DATA_DIR / "extracted_items"
CLEANED_ITEMS_DIR = DATA_DIR / "cleaned_items"

for folder in [EXTRACTED_ITEMS_DIR, CLEANED_ITEMS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


## 2. Select Items and Explain Why

Choose items because they support your product questions. The example below focuses on risk, strategy, and financial discussion.


In [ ]:
selected_items = pd.DataFrame(
    [
        {"item_number": "1", "item_name": "Business", "why_selected": "Explains products, customers, and industry position."},
        {"item_number": "1A", "item_name": "Risk Factors", "why_selected": "Supports risk analysis and volatility comparisons."},
        {"item_number": "7", "item_name": "MD&A", "why_selected": "Connects management language to financial performance."},
    ]
)

selected_items.to_csv(OUTPUTS_DIR / "selected_items_rationale.csv", index=False)
selected_items


## 3. Cleaning Function

This function removes common HTML artifacts, script/style content, repeated whitespace, and table-of-contents-like line noise. It preserves the main text so retrieval still has useful evidence.


In [ ]:
import re
from html import unescape

def clean_sec_item_text(raw_text):
    text = unescape(str(raw_text))
    text = re.sub(r"<script.*?</script>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<style.*?</style>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"Table of Contents", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bITEM\s+\d+[A-Z]?\.?\s*$", " ", text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

sample_html = "<html><body><p>Item 1A. Risk Factors</p><p>Cybersecurity risk increased.</p></body></html>"
clean_sec_item_text(sample_html)


## 4. Clean All Extracted Item Files

This code assumes Step 1 saved item files in `data/extracted_items/`. It writes cleaned `.txt` files to `data/cleaned_items/` and creates a metadata table.


In [ ]:
from pathlib import Path

metadata_path = OUTPUTS_DIR / "extracted_item_metadata.csv"
if metadata_path.exists():
    extracted_metadata = pd.read_csv(metadata_path)
else:
    extracted_metadata = pd.DataFrame(columns=["extracted_file"])

records = []

for _, item in extracted_metadata.iterrows():
    source_name = item["extracted_file"]
    source_path = EXTRACTED_ITEMS_DIR / source_name
    if not source_path.exists():
        continue

    raw_text = source_path.read_text(encoding="utf-8", errors="ignore")
    cleaned_text = clean_sec_item_text(raw_text)

    cleaned_name = source_path.stem + "_cleaned.txt"
    cleaned_path = CLEANED_ITEMS_DIR / cleaned_name
    cleaned_path.write_text(cleaned_text, encoding="utf-8")

    record = item.to_dict()
    record.update(
        {
            "source_file": item.get("source_file", source_name),
            "extracted_file": source_name,
            "cleaned_file": cleaned_name,
            "raw_char_count": len(raw_text),
            "cleaned_char_count": len(cleaned_text),
            "raw_word_count": len(raw_text.split()),
            "cleaned_word_count": len(cleaned_text.split()),
            "cleaning_retention_rate": len(cleaned_text.split()) / max(len(raw_text.split()), 1),
        }
    )
    records.append(record)

cleaned_metadata = pd.DataFrame(records)
cleaned_metadata.to_csv(OUTPUTS_DIR / "cleaned_item_metadata.csv", index=False)

display(cleaned_metadata.head())
display(cleaned_metadata.groupby(["ticker", "item_number"]).size().unstack(fill_value=0) if not cleaned_metadata.empty else cleaned_metadata)


## 5. EDA Checks

The goal of EDA is to prove that the corpus is usable. At minimum, inspect document lengths and the most frequent terms after cleaning.


In [ ]:
from collections import Counter

if cleaned_metadata.empty:
    print("No cleaned files found yet. Run Step 1 extraction first, then rerun this notebook.")
else:
    display(cleaned_metadata["cleaned_word_count"].describe().to_frame("word_count_summary"))

    all_words = []
    stop_words = {"the", "and", "of", "to", "in", "for", "our", "we", "a", "an", "is", "are", "with", "as", "by"}

    for cleaned_file in cleaned_metadata["cleaned_file"]:
        text = (CLEANED_ITEMS_DIR / cleaned_file).read_text(encoding="utf-8", errors="ignore").lower()
        words = re.findall(r"[a-z]{3,}", text)
        all_words.extend([w for w in words if w not in stop_words])

    top_terms = pd.DataFrame(Counter(all_words).most_common(25), columns=["term", "count"])
    top_terms.to_csv(OUTPUTS_DIR / "top_terms_step2.csv", index=False)
    display(top_terms)


## Step 2 Interpretation Prompt

After the code, add a short markdown paragraph explaining what changed from raw to cleaned text. Mention any missing companies, missing years, unusually short files, or terms that look like boilerplate.
